# 看懂 Agent 每一步怎麼跑這份教材會先準備模組，再單獨準備執行紀錄工具，最後才執行工作流程。重點是看懂每一步怎麼留下資料。

## 在 Colab 準備環境先準備 SDK 執行環境。

In [ ]:
from pathlib import Pathimport os, sys, subprocessif not Path('agentic_sdk').exists():    if not Path('Agentic-SDK').exists():        subprocess.run(['git', 'clone', 'https://github.com/R300-AI/Agentic-SDK.git'], check=True)    os.chdir('Agentic-SDK')subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)print('Agentic SDK ready')

## 載入這次要觀察的流程元件先載入工作流程與三個基礎模組。這份教材的主角不是模組參數，而是執行過程。

In [ ]:
from agentic_sdk import Workflowfrom agentic_sdk.modules import DirectAnswerAction, KeywordRetrieve, PassThroughPerceive

## 先準備可觀察的資料和模組把資料和模組分開宣告，待會看紀錄時比較容易知道每一步在處理哪一塊。

In [ ]:
knowledge_items = [    {'keywords': ['sdk'], 'content': 'Agentic SDK 的流程可以被逐步觀察。'},]perceive = PassThroughPerceive()retrieve = KeywordRetrieve(items=knowledge_items)action = DirectAnswerAction()

## 單獨準備執行紀錄`event_callback` 會在每個節點開始與結束時被呼叫。這裡先準備一個清單，把每次事件收起來。

In [ ]:
events = []def collect_event(event):    events.append({        'phase': event.get('phase'),        'module': event.get('module'),        'next_module': event.get('next_module'),        'visit_count': event.get('visit_count'),    })

## 把可觀察的流程組起來模組和紀錄工具都準備好後，再建立完整工作流程。

In [ ]:
workflow = Workflow(    workflow_name='可追蹤問答 Agent',    perceive=perceive,    retrieve=retrieve,    action=action,)

## 執行並收集紀錄這次執行時多傳入 `event_callback`，流程每跑一步就會把紀錄丟進 `events`。

In [ ]:
result = workflow.run('請介紹 SDK 的追蹤方式', event_callback=collect_event)print(result.final_message)

## 先看每一步的開始與結束這個輸出可以幫你看流程走過哪些節點，以及每一步結束後下一步要去哪裡。

In [ ]:
for index, event in enumerate(events, start=1):    print(index, event)

## 再看流程留下的資料`visit_counts` 告訴你各節點跑了幾次；`entities` 是節點之間交換的中間資料。

In [ ]:
print('跑過哪些節點:', result.visit_counts)print('中間資料:', result.entities)